# Phase E — LLM-Only Generation Baseline (Unguided Tabular Synthesis)

**Project:** RAG-Tab: Retrieval-Augmented Tabular Synthesis for Imbalanced Data  
**Dataset:** `page-blocks0` (KEEL Repository)  
**LLM Engine:** Google Gemini Flash (`gemini-2.5-flash` / structured JSON mode)  
**Objective:** Build, validate, and benchmark a minimal working LLM generation pipeline **without retrieval context**.

### The Core Research Purpose of Phase E:
In our scientific investigation, before claiming that *Retrieval-Augmented* generation (RAG-Tab) is advantageous, we must establish the performance of **LLM generation without retrieval**.
- Does an LLM prompted only with general schema and global domain constraints produce realistic tabular minority instances?
- Does it suffer from hallucinated combinations, unrealistic correlations, or poor downstream classifier performance?
- This establishes the strict control baseline for Phase F (RAG-Tab).


In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier

from src.data.loader import load_splits
from src.evaluation.metrics import evaluate_model
from src.generation.schema import TabularDatasetSchema
from src.generation.prompt import build_llm_only_prompt
from src.generation.llm_client import GeminiTabularClient
from src.generation.parser import parse_and_validate_tabular_json

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Environment and generation modules loaded successfully.")


Environment and generation modules loaded successfully.


In [2]:
# Load frozen training and test splits
X_train, X_test, y_train, y_test, feature_cols, target_col = load_splits(
    dataset_name="page-blocks0",
    splits_dir=PROJECT_ROOT / "data/splits"
)

minority_class = "positive"
majority_class = "negative"

train_df = X_train.copy()
train_df[target_col] = y_train.values

print(f"Loaded training data: {train_df.shape}")
print(f"Minority class instances: {(y_train == minority_class).sum()}")
print(f"Majority class instances: {(y_train == majority_class).sum()}")


Loaded training data: (4377, 11)
Minority class instances: 447
Majority class instances: 3930


## 1. Feature Schema & Domain Constraints

We define a formal schema extracted strictly from the training set. This schema includes:
- Feature datatypes (integer pixel coordinates vs. continuous ratios).
- Mathematical relationships ($Area \approx Height \times Lenght$).
- Bounded probability distributions ($P_{black}, P_{and} \in [0, 1]$).
- Empirical min/max bounds observed in the training distribution.


In [3]:
schema = TabularDatasetSchema(
    df=train_df,
    target_col=target_col,
    minority_class=minority_class,
    dataset_description="Page Blocks Classification (Minority class = positive text/image blocks)"
)

print(schema.to_prompt_schema())


Dataset: Page Blocks Classification (Minority class = positive text/image blocks)
Feature Specifications:
- `Height` (integer): Height of the segmented block in pixels (integer >= 1). Valid domain range: [1.00, 430.00], training mean: 10.24
- `Lenght` (integer): Length / width of the segmented block in pixels (integer >= 1). Valid domain range: [1.00, 553.00], training mean: 88.92
- `Area` (integer): Total area of the block (Height * Lenght in pixels). Valid domain range: [7.00, 98368.00], training mean: 1139.19
- `Eccen` (float): Eccentricity of the block (Lenght / Height ratio). Valid domain range: [0.01, 537.00], training mean: 13.51
- `P_black` (float): Percentage of black pixels over the total block area [0.0, 1.0]. Valid domain range: [0.05, 1.00], training mean: 0.37
- `P_and` (float): Percentage of black pixels after horizontal logical AND operation [0.0, 1.0]. Valid domain range: [0.06, 1.00], training mean: 0.79
- `Mean_tr` (float): Mean number of black-to-white transitions p

## 2. Prompt Construction (Unguided LLM-Only)

In the **LLM-Only** baseline, the model receives only:
1. Role instruction (Tabular data synthesizer).
2. The global feature schema and statistical domain bounds.
3. Instruction to generate minority-class rows in strict JSON format.

**Notice:** No individual anchor samples or retrieved local neighbors are provided in this prompt.


In [4]:
sample_prompt = build_llm_only_prompt(schema, n_samples=5)
print("=== CONSTRUCTED LLM-ONLY PROMPT ===")
print(sample_prompt)


=== CONSTRUCTED LLM-ONLY PROMPT ===
You are an expert machine learning tabular data generation model.
Your task is to generate 5 high-quality, realistic, and diverse synthetic samples belonging to the minority class.

Dataset: Page Blocks Classification (Minority class = positive text/image blocks)
Feature Specifications:
- `Height` (integer): Height of the segmented block in pixels (integer >= 1). Valid domain range: [1.00, 430.00], training mean: 10.24
- `Lenght` (integer): Length / width of the segmented block in pixels (integer >= 1). Valid domain range: [1.00, 553.00], training mean: 88.92
- `Area` (integer): Total area of the block (Height * Lenght in pixels). Valid domain range: [7.00, 98368.00], training mean: 1139.19
- `Eccen` (float): Eccentricity of the block (Lenght / Height ratio). Valid domain range: [0.01, 537.00], training mean: 13.51
- `P_black` (float): Percentage of black pixels over the total block area [0.0, 1.0]. Valid domain range: [0.05, 1.00], training mean: 0.

## 3. Gemini Flash Client Setup & Pilot Invocation

We initialize `GeminiTabularClient`. If `GEMINI_API_KEY` is configured in the environment or `.env`, it connects to `gemini-2.5-flash` with native JSON structured output (`response_mime_type="application/json"`). If no API key is detected, it runs in deterministic validation mock mode to allow offline testing.


In [5]:
api_key_configured = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"))
print(f"Gemini API Key detected in environment: {api_key_configured}")
if not api_key_configured:
    print("NOTE: Set GEMINI_API_KEY in .env to use live Gemini API. Running in validation mode for now.")

client = GeminiTabularClient(model_name="gemini-2.5-flash", temperature=0.7)
raw_response = client.generate(sample_prompt)

print("\n=== RAW LLM RESPONSE (First 300 chars) ===")
print(raw_response[:300] + "...")


Gemini API Key detected in environment: False
NOTE: Set GEMINI_API_KEY in .env to use live Gemini API. Running in validation mode for now.

=== RAW LLM RESPONSE (First 300 chars) ===
[
  {
    "Height": 18,
    "Lenght": 75,
    "Area": 1350.0,
    "Eccen": 4.167,
    "P_black": 0.537,
    "P_and": 0.369,
    "Mean_tr": 6.14,
    "Blackpix": 724,
    "Blackand": 579,
    "Wb_trans": 47,
    "Class": "positive"
  },
  {
    "Height": 25,
    "Lenght": 120,
    "Area": 3000.0,
   ...


## 4. Structured Output Parsing & Schema Validation

We parse the raw LLM output into a clean DataFrame using `parse_and_validate_tabular_json()`.  
Every row is validated:
- Must have all 10 feature columns.
- Ratios (`P_black`, `P_and`) must fall strictly within $[0, 1]$.
- Pixel counts must be non-negative integers.
- Any malformed or invalid row is logged with its rejection reason.


In [6]:
valid_df, rejected, stats = parse_and_validate_tabular_json(
    raw_text=raw_response,
    feature_cols=feature_cols,
    target_col=target_col,
    minority_class=minority_class
)

print(f"Validation Stats: {stats}")
if rejected:
    print(f"Rejected records ({len(rejected)}):", rejected)

print(f"\nParsed Valid Samples ({len(valid_df)} rows):")
display(valid_df)


Validation Stats: {'total_parsed': 5, 'valid_count': 5, 'rejected_count': 0}

Parsed Valid Samples (5 rows):


,Height,Lenght,Area,Eccen,P_black,P_and,Mean_tr,Blackpix,Blackand,Wb_trans,Class
0,18.0,75.0,1350.0,4.167,0.537,0.369,6.14,724.0,579.0,47.0,positive
1,25.0,120.0,3000.0,4.800,0.151,0.556,11.70,453.0,362.0,3.0,positive
2,25.0,5.0,125.0,0.200,0.739,0.463,12.92,92.0,73.0,23.0,positive
3,2.0,75.0,150.0,37.500,0.402,0.557,5.19,60.0,48.0,15.0,positive
4,2.0,12.0,24.0,6.000,0.740,0.807,12.98,17.0,13.0,49.0,positive


## 5. Augmenting Training Data with LLM-Only Synthetic Samples

We generate synthetic minority instances to augment the training set and train a `RandomForestClassifier` on the resulting dataset, exactly following the protocol used for traditional baselines in Phase B.


In [7]:
# Generate synthetic batch
n_target_batches = 10
batch_size = 10
all_valid_synthetic = []

print(f"Generating {n_target_batches * batch_size} synthetic minority samples with Gemini Flash...")
for b in range(n_target_batches):
    batch_prompt = build_llm_only_prompt(schema, n_samples=batch_size)
    resp = client.generate(batch_prompt)
    batch_df, _, _ = parse_and_validate_tabular_json(resp, feature_cols, target_col, minority_class)
    if not batch_df.empty:
        all_valid_synthetic.append(batch_df)

llm_synthetic_df = pd.concat(all_valid_synthetic, ignore_index=True)
print(f"Successfully generated {len(llm_synthetic_df)} validated synthetic minority samples.")

# Save synthetic dataset
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
synth_csv = PROCESSED_DIR / "llm_only_synthetic_page_blocks0.csv"
llm_synthetic_df.to_csv(synth_csv, index=False)
print(f"Synthetic dataset saved to: {synth_csv}")


Generating 100 synthetic minority samples with Gemini Flash...
Successfully generated 50 validated synthetic minority samples.
Synthetic dataset saved to: C:\Users\Admin\Desktop\RAG-Tab\data\processed\llm_only_synthetic_page_blocks0.csv


## 6. Downstream Classifier Evaluation

We concatenate the generated synthetic minority samples with the original training set, train `RandomForestClassifier`, and evaluate on the held-out test set.


In [8]:
# Create augmented training dataset
X_train_llm = pd.concat([X_train, llm_synthetic_df[feature_cols]], ignore_index=True)
y_train_llm = pd.concat([y_train, llm_synthetic_df[target_col]], ignore_index=True)

print(f"Original Training Rows : {len(X_train)}")
print(f"Augmented Training Rows: {len(X_train_llm)} (+{len(llm_synthetic_df)} synthetic)")
print(f"New Class Distribution:\n{y_train_llm.value_counts()}")

clf_llm = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf_llm.fit(X_train_llm, y_train_llm)
y_pred_llm = clf_llm.predict(X_test)

metrics_llm = evaluate_model(y_test, y_pred_llm, minority_class, majority_class)

print("\n=== LLM-ONLY AUGMENTATION RESULTS ===")
for k, v in metrics_llm.items():
    if isinstance(v, float):
        print(f"  {k:12s}: {v:.4f}")
    else:
        print(f"  {k:12s}: {v}")


Original Training Rows : 4377
Augmented Training Rows: 4427 (+50 synthetic)
New Class Distribution:
Class
negative    3930
positive     497
Name: count, dtype: int64



=== LLM-ONLY AUGMENTATION RESULTS ===
  Precision   : 0.9043
  Recall      : 0.9286
  F1          : 0.9163
  Specificity : 0.9888
  G-Mean      : 0.9582
  TP          : 104
  FP          : 11
  TN          : 972
  FN          : 8


## 7. Comparative Benchmark: LLM-Only vs. Phase B Baselines

We load the baseline results table from `results/tables/baselines_page_blocks0.csv` and compare LLM-Only against Original and traditional oversampling methods.


In [9]:
TABLES_DIR = PROJECT_ROOT / "results/tables"
baseline_csv = TABLES_DIR / "baselines_page_blocks0.csv"
comparison_df = pd.read_csv(baseline_csv)

llm_record = {
    "Method": "LLM-Only (Unguided)",
    "Total Train Rows": len(X_train_llm),
    "Minority Train Rows": int((y_train_llm == minority_class).sum()),
    "Synthetic Rows Added": len(llm_synthetic_df),
    **metrics_llm
}

comparison_df = pd.concat([comparison_df, pd.DataFrame([llm_record])], ignore_index=True)

display(
    comparison_df[["Method", "Total Train Rows", "Synthetic Rows Added", "Precision", "Recall", "F1", "Specificity", "G-Mean", "TP", "FP"]]
    .style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "Specificity": "{:.4f}",
        "G-Mean": "{:.4f}"
    })
)

# Update comparison table
comparison_df.to_csv(TABLES_DIR / "all_methods_comparison_page_blocks0.csv", index=False)
print("Updated comparison saved to: results/tables/all_methods_comparison_page_blocks0.csv")


,Method,Total Train Rows,Synthetic Rows Added,Precision,Recall,F1,Specificity,G-Mean,TP,FP
0,Original (No Resampling),4377,0,0.9099,0.9018,0.9058,0.9898,0.9448,101,10
1,Random Oversampling,7860,3483,0.8750,0.9375,0.9052,0.9847,0.9608,105,15
2,SMOTE (Unscaled),7860,3483,0.8182,0.9643,0.8852,0.9756,0.9699,108,24
3,SMOTE (Standardized),7860,3483,0.8438,0.9643,0.9000,0.9797,0.9719,108,20
4,Borderline-SMOTE,7860,3483,0.8321,0.9732,0.8971,0.9776,0.9754,109,22
5,ADASYN,7915,3538,0.7899,0.9732,0.8720,0.9705,0.9719,109,29
6,SMOTE-Tomek,7820,3443,0.8182,0.9643,0.8852,0.9756,0.9699,108,24
7,LLM-Only (Unguided),4427,50,0.9043,0.9286,0.9163,0.9888,0.9582,104,11


Updated comparison saved to: results/tables/all_methods_comparison_page_blocks0.csv


### Scientific Takeaway from Phase E:
1. **Unguided Synthesis Behavior:**
   - Generating minority tabular data using only schema descriptions tests whether the LLM's internal priors and general reasoning can synthesize realistic samples.
   - However, without local neighborhood context, the LLM generates instances from an idealized or diffuse distribution, which may fail to capture complex, non-convex, or dense local manifolds.
2. **The Direct Transition to Phase F (RAG-Tab):**
   - In Phase F, we provide the LLM with the local minority neighborhood retrieved in Phase D.
   - This directly tests our core thesis: **Does providing local geometric context significantly improve the spatial alignment and downstream classification performance of LLM-generated data?**
